## LightGBM — Leaf-Wise Growth

### 1. Simple explanation

A **Decision Tree** needs to decide **which node to split next**.

There are two common strategies:

- **Level-wise growth** → split all nodes at the same depth before moving deeper.
- **Leaf-wise growth** → choose the **single leaf whose split gives the greatest improvement**, and split that leaf.

LightGBM uses **leaf-wise growth**.

The key idea is:

> Instead of asking “Which nodes are at the next level?”, LightGBM asks “Which leaf can improve the model the most if I split it?”

---

### 2. Level-wise vs Leaf-wise

Suppose the current tree looks like this:

```text
             Root
            /    \
           A      B
         /  \    /  \
        C    D  E    F
```

In **level-wise growth**, after splitting the root, it considers both `A` and `B` at the next level.

So the tree grows roughly like:

```text
Depth 0       Root
               |
Depth 1      A     B
             |     |
Depth 2    C  D   E  F
```

The important point is that nodes at the same depth are processed together.

---

### 3. Leaf-wise growth

In LightGBM, suppose splitting the leaves gives these improvements:

| Leaf | Gain from splitting |
|---|---:|
| A | 12 |
| B | 5 |
| C | 3 |
| D | 18 |

LightGBM chooses:

$$\text{Best leaf}=\arg\max_{\text{leaf}} \text{Gain}$$

So it chooses **D**, because its split gives gain $18$.

It does **not** care that another leaf might be at a shallower depth.

The tree could therefore become unbalanced:

```text
          Root
         /    \
        A      B
       / \
      C   D
         / \
        ... ...
```

If another leaf later has the highest gain, that leaf is split next.

---

## 4. Why does LightGBM do this?

The purpose is to obtain the **largest possible reduction in the objective/loss at each tree-growing step**.

Suppose we have several leaves.

For each leaf, LightGBM evaluates possible splits and determines how much the split improves the objective.

Conceptually:

$$\text{Gain}=\text{Objective before split}-\text{Objective after split}$$

The leaf with the highest gain is selected.

Therefore:

```text
Find all candidate leaf splits
          ↓
Calculate split gains
          ↓
Choose highest-gain leaf
          ↓
Split that leaf
          ↓
Repeat
```

This is why leaf-wise growth can produce a **more accurate tree with fewer leaves** than level-wise growth in many situations.

---

## 5. Mathematical intuition

For a boosting tree, LightGBM evaluates how much a candidate split improves the objective.

For a simplified second-order boosting formulation, the gain can be expressed as:

$$\text{Gain}=\frac{1}{2}\left(\frac{G_L^2}{H_L+\lambda}+\frac{G_R^2}{H_R+\lambda}-\frac{G^2}{H+\lambda}\right)-\gamma$$

Where:

- $G$ = sum of gradients in the parent
- $H$ = sum of Hessians in the parent
- $G_L,H_L$ = gradient and Hessian sums for the left child
- $G_R,H_R$ = gradient and Hessian sums for the right child
- $\lambda$ = L2 regularization
- $\gamma$ = minimum gain required for making a split

You don't need to memorize the derivation here. The important idea is:

> **Every candidate split gets a gain, and leaf-wise growth chooses the leaf/split with the highest gain.**

This connects directly to what you already learned in XGBoost: the tree is looking for a split that produces the best improvement.

---

## 6. Small example

Suppose the current tree has three leaves:

```text
             Root
            /    \
           A      B
                 / \
                C   D
```

Possible best split gains:

| Leaf | Best possible split gain |
|---|---:|
| A | 4 |
| C | 15 |
| D | 8 |

LightGBM chooses **C**:

$$\max(4,15,8)=15$$

So:

```text
             Root
            /    \
           A      B
                 / \
                C   D
               / \
             C1   C2
```

Now it evaluates the new leaves again.

Suppose:

| Leaf | Best gain |
|---|---:|
| A | 4 |
| C1 | 3 |
| C2 | 20 |
| D | 8 |

It chooses **C2**, because its gain is $20$.

Notice what happened:

```text
C
↓
C1, C2
↓
C2 is selected
↓
C2 is split again
```

The tree becomes deeper in one particular region.

That's the defining behavior of **leaf-wise growth**.

---

## 7. Why can it be faster / more accurate?

### More targeted growth

Level-wise growth may split several nodes even when some of those splits provide relatively little improvement.

Leaf-wise growth focuses on the **most useful split**.

So for a fixed number of leaves, it can often achieve a larger reduction in loss.

### Example

Suppose:

```text
Level-wise:

        Root
       /    \
      A      B
     / \    / \
```

Even if:

```text
Gain(A) = 20
Gain(B) = 2
```

both may be expanded.

Leaf-wise instead focuses on:

```text
A → gain 20
```

and spends the next split where it provides the most benefit.

---

# 8. Main drawback — overfitting

This is extremely important.

Because LightGBM always chooses the leaf with the highest gain, it can repeatedly focus on a small region of the data.

You might get:

```text
                 Root
                /    \
               A      B
                    /  \
                   C    D
                       / \
                      E   F
                         / \
                        ...
```

One branch can become **very deep** while other branches remain shallow.

If the tree keeps making highly specific splits, it can start fitting noise.

Therefore:

> **Leaf-wise growth generally has a stronger tendency to overfit than level-wise growth if the tree is allowed to grow without sufficient constraints.**

This is why parameters such as `num_leaves`, `max_depth`, and `min_child_samples` become important when controlling LightGBM complexity.

---

## 9. Level-wise vs Leaf-wise — core difference

| Level-wise | Leaf-wise |
|---|---|
| Splits nodes level by level | Splits best leaf |
| More balanced tree | Can create unbalanced tree |
| Considers depth structure | Considers split gain |
| Usually less aggressive | More aggressive |
| Lower overfitting tendency | Higher overfitting tendency |
| Used traditionally by many tree algorithms | Core strategy of LightGBM |

### The one thing to remember

**Level-wise:**

> “Complete this level.”

**Leaf-wise:**

> “Find the leaf whose split improves the model the most.”

LightGBM's leaf-wise strategy is essentially **greedy optimization over leaves**: at every step, it chooses the split with the greatest gain. Pasted text

## LightGBM — Histogram-Based Learning

### 1. Simple explanation

Normally, when a decision tree wants to find the best split for a numerical feature, it can examine many possible threshold values.

For example:

```text
Age = [21, 24, 27, 31, 35, 42]
```

Possible splits could be:

```text
Age < 22.5
Age < 25.5
Age < 29
Age < 33
Age < 38.5
```

If the dataset is very large and has many features, checking many thresholds becomes expensive.

**LightGBM solves this using histogram-based learning.**

Instead of considering every individual feature value, LightGBM groups continuous values into a fixed number of **bins**.

---

## 2. What is a histogram?

Suppose a feature contains:

```text
Age = [18, 21, 23, 25, 31, 34, 38, 42]
```

Instead of treating every value separately, LightGBM might divide the range into 4 bins:

```text
Bin 1 → 18–23
Bin 2 → 24–29
Bin 3 → 30–35
Bin 4 → 36–42
```

The actual values are therefore represented by their **bin indices**:

```text
18 → Bin 1
21 → Bin 1
23 → Bin 1
25 → Bin 2
31 → Bin 3
34 → Bin 3
38 → Bin 4
42 → Bin 4
```

So instead of testing every possible numerical threshold, LightGBM tests **bin boundaries**.

---

## 3. Why does this improve efficiency?

Suppose a feature has:

$$1,000,000$$

different observations.

Without binning, there can potentially be a very large number of candidate thresholds.

With:

$$\text{max\_bin}=255$$

the feature is represented using at most approximately 255 bins.

Therefore, instead of considering a huge number of individual values, the algorithm works with a much smaller set of possible split points.

The basic idea is:

```text
Raw feature values
        ↓
Create bins
        ↓
Calculate gradient/hessian statistics per bin
        ↓
Scan bins
        ↓
Find best split
```

---

## 4. How does this connect to XGBoost?

You already learned that XGBoost uses gradients and Hessians.

The same general idea applies here.

For each training sample, we have:

$$g_i=\frac{\partial l(y_i,\hat{y}_i)}{\partial\hat{y}_i}$$

and

$$h_i=\frac{\partial^2 l(y_i,\hat{y}_i)}{\partial\hat{y}_i^2}$$

Instead of repeatedly examining individual samples, LightGBM aggregates these values into bins.

For a particular bin:

$$G_{\text{bin}}=\sum_{i\in\text{bin}}g_i$$

$$H_{\text{bin}}=\sum_{i\in\text{bin}}h_i$$

So each bin stores aggregated gradient and Hessian information.

---

## 5. Finding a split using bins

Suppose we have:

| Bin | Samples | $G$ | $H$ |
|---|---:|---:|---:|
| 1 | 20 | -8 | 20 |
| 2 | 15 | -5 | 15 |
| 3 | 30 | 12 | 30 |
| 4 | 10 | 6 | 10 |

LightGBM can consider:

```text
Split after Bin 1
Split after Bin 2
Split after Bin 3
```

For example, if splitting after Bin 2:

```text
Left  → Bin 1 + Bin 2
Right → Bin 3 + Bin 4
```

The aggregated statistics are:

$$G_L=G_1+G_2$$

$$H_L=H_1+H_2$$

and:

$$G_R=G_3+G_4$$

$$H_R=H_3+H_4$$

Then LightGBM calculates the split gain.

So the expensive part of searching through individual values is greatly reduced.

---

## 6. Why histograms save memory too

There is another important advantage.

Instead of repeatedly working with full-precision feature values during split searching, LightGBM can represent features using compact bin indices.

For example:

```text
Original:

18.234
18.891
19.102
21.456
...
```

becomes something like:

```text
3
3
4
7
...
```

when values are mapped to bins.

This reduces the amount of information that needs to be processed during tree construction.

Therefore histogram-based learning improves:

- **training speed**
- **memory efficiency**
- **split-search efficiency**

---

## 7. What does `max_bin` control?

The important parameter here is:

```python
max_bin
```

It controls the maximum number of bins used for numerical features.

For example:

```python
LGBMClassifier(max_bin=255)
```

A larger `max_bin` means:

```text
More bins
→ more precise representation
→ potentially better split precision
→ more computation/memory
```

A smaller `max_bin` means:

```text
Fewer bins
→ faster computation
→ lower memory usage
→ potentially less precise splits
```

So there is a trade-off.

---

## 8. Important limitation

Histogram binning is an **approximation**.

Suppose the exact best threshold is:

$$x<37.42$$

but the bin boundaries only allow:

```text
35
40
```

Then LightGBM cannot test exactly $37.42$ as a threshold.

It will work with the available bin boundary.

Therefore:

> Histogram-based learning gains efficiency by sacrificing some split precision.

In practice, the reduction in computation is often worth this approximation.

---

## 9. The key mechanism

Remember this flow:

```text
Continuous feature
       ↓
Discretize into bins
       ↓
Aggregate gradients + Hessians
       ↓
Evaluate bin boundaries
       ↓
Calculate split gain
       ↓
Choose best split
```

And this is different from **leaf-wise growth**:

- **Leaf-wise growth** answers: **which leaf should I split?**
- **Histogram-based learning** answers: **how can I efficiently search for the best split inside that leaf?**

So these are two different parts of LightGBM working together.

## LightGBM — GOSS (Gradient-based One-Side Sampling)

### 1. Simple explanation

**GOSS = Gradient-based One-Side Sampling.**

The main problem it solves is:

> LightGBM may have a very large number of training samples, so using every sample for every tree split can be expensive.

GOSS reduces the number of samples used during tree construction.

But it does **not randomly remove samples**.

Instead, it keeps samples based on their **gradient magnitude**.

The key idea:

> Keep samples with **large gradients**, and randomly sample only some samples with **small gradients**.

---

### 2. Why are gradients important?

Recall from XGBoost:

$$g_i=\frac{\partial l(y_i,\hat{y}_i)}{\partial\hat{y}_i}$$

Here $g_i$ tells us how much the current prediction needs to be corrected for sample $i$.

A **large gradient magnitude** means:

$$|g_i|\text{ is large}$$

which generally indicates that the current model is doing relatively poorly on that sample.

A **small gradient magnitude** means the current prediction is already relatively good.

So GOSS considers large-gradient samples more important for finding useful splits.

---

## 3. What does GOSS actually do?

Suppose there are 10,000 samples.

Imagine their gradients are sorted by magnitude:

```text
Largest gradient
      ↓
[large] [large] [large] ... [large]
                         ↓
                  important samples

[small] [small] [small] ... [small]
                         ↓
                  less important
```

GOSS does two things:

### Step 1 — Keep large-gradient samples

It keeps a large proportion of samples having the largest $|g_i|$.

For example:

$$a=0.2$$

means approximately the top 20% large-gradient samples are retained.

### Step 2 — Randomly sample small-gradient samples

From the remaining samples, GOSS randomly selects only a fraction.

For example:

$$b=0.1$$

means 10% of the remaining small-gradient samples are selected.

Therefore, instead of using all 10,000 samples, it might use far fewer.

---

## 4. But there is a problem

If we simply throw away most small-gradient samples, the distribution of the training data becomes biased.

For example:

```text
Original:
Large gradient → 20%
Small gradient → 80%
```

After sampling:

```text
Large gradient → 20% retained
Small gradient → only 10% of them retained
```

The selected dataset now contains a much larger proportion of high-gradient samples.

This could make the calculated gradients and Hessians misleading.

So GOSS uses **re-weighting**.

---

## 5. Re-weighting the sampled small-gradient observations

Suppose:

- $a$ = fraction of large-gradient samples kept
- $b$ = fraction of small-gradient samples sampled

For the randomly selected small-gradient samples, their gradient contribution is multiplied by:

$$\frac{1-a}{b}$$

This compensates for the small-gradient samples that were discarded.

### Example

Suppose:

$$a=0.2$$

and:

$$b=0.1$$

Then the weight becomes:

$$\frac{1-a}{b}=\frac{0.8}{0.1}=8$$

So every selected small-gradient sample receives a factor of **8** when calculating the relevant statistics.

---

## 6. Why does this make sense?

Suppose there were originally 800 small-gradient samples.

We randomly keep only:

$$800\times0.1=80$$

samples.

Those 80 samples are supposed to represent the original 800.

Therefore, each selected sample represents approximately:

$$\frac{800}{80}=10$$

original samples.

The exact GOSS correction is based on the sampling fractions and gives the factor:

$$\frac{1-a}{b}$$

This allows LightGBM to preserve the influence of the discarded small-gradient observations more accurately.

---

## 7. How GOSS connects to histogram learning

This is important because GOSS isn't a completely separate tree-building algorithm.

The flow is:

```text
Training data
     ↓
Calculate gradients
     ↓
GOSS sampling
     ↓
Keep large-gradient samples
+ randomly sample small-gradient samples
     ↓
Build histograms using selected samples
     ↓
Find best split
     ↓
Leaf-wise tree growth
```

So:

**Leaf-wise growth** determines **which leaf to split**.

**Histogram-based learning** makes split searching efficient.

**GOSS** reduces the number of samples involved in the computation.

These mechanisms work together.

---

## 8. Small numerical example

Suppose we have 10 samples:

| Sample | $|g_i|$ |
|---|---:|
| 1 | 10 |
| 2 | 9 |
| 3 | 8 |
| 4 | 7 |
| 5 | 6 |
| 6 | 3 |
| 7 | 2 |
| 8 | 1.5 |
| 9 | 1 |
| 10 | 0.5 |

Suppose:

$$a=0.4$$

So GOSS keeps the top 40%:

```text
Samples 1, 2, 3, 4
```

Now there are 6 small-gradient samples:

```text
5, 6, 7, 8, 9, 10
```

Suppose:

$$b=0.5$$

GOSS randomly keeps 50% of these, say:

```text
5, 6, 8
```

So the tree construction uses:

```text
Large-gradient:
1, 2, 3, 4

Sampled small-gradient:
5, 6, 8
```

Instead of all 10 samples, only 7 are directly used.

The selected small-gradient samples are re-weighted using:

$$\frac{1-a}{b}=\frac{0.6}{0.5}=1.2$$

---

## 9. Why GOSS can improve training speed

Without GOSS:

```text
All samples
   ↓
Histogram construction
   ↓
Split evaluation
```

With GOSS:

```text
All samples
   ↓
Gradient calculation
   ↓
Select important samples
   ↓
Histogram construction on reduced data
   ↓
Split evaluation
```

Fewer samples need to participate in the expensive tree-building calculations.

Therefore:

> **GOSS reduces computational cost while trying to preserve the samples that are most useful for improving the model.**

---

## 10. Most important point to remember

Don't think of GOSS as:

> "Randomly sample the dataset."

That's incorrect.

Think:

> **GOSS keeps high-gradient samples because they contain more information about where the model currently makes large errors, and randomly samples low-gradient samples to reduce computation.**

So:

$$\boxed{\text{Large gradients → keep; Small gradients → sample}}$$

with **re-weighting** to compensate for the discarded small-gradient samples.

### Why does keeping all the large gradients and skipping most of the small ones introduce bias, and how exactly does the correction factor fix that?

### 1. Start with the original dataset

Suppose we have **100 samples**:

```text
Large-gradient samples  = 20
Small-gradient samples  = 80
```

So originally:

```text
Large = 20%
Small = 80%
```

Imagine, for simplicity, that every sample contributes `1` to some statistic.

Therefore the original total contribution is:

$$20(1)+80(1)=100$$

---

### 2. Now GOSS samples

GOSS keeps **all 20 large-gradient samples**.

But suppose it keeps only **10% of the small-gradient samples**:

$$80\times0.10=8$$

So now the working dataset contains:

```text
Large = 20
Small = 8
Total = 28
```

Look at the proportions now:

$$\text{Large proportion}=\frac{20}{28}=71.4\%$$

$$\text{Small proportion}=\frac{8}{28}=28.6\%$$

But originally:

```text
Large = 20%
Small = 80%
```

So the selected dataset is **heavily over-representing large-gradient samples**.

That's what we mean by **sampling bias** here.

---

# 3. How does the factor fix it?

We need those 8 selected small-gradient samples to **represent the 80 small-gradient samples that existed originally**.

We kept:

$$8$$

but they represent:

$$80$$

Therefore, each selected sample should represent:

$$\frac{80}{8}=10$$

small-gradient samples.

So we multiply their contribution by **10**.

For example, suppose each small-gradient sample has:

$$g_i=2$$

Without correction:

```text
8 samples × 2 = 16
```

But the original 80 samples would have contributed:

```text
80 × 2 = 160
```

After weighting:

$$8\times(2\times10)=160$$

Now we've recovered the contribution that the discarded samples would have provided.

---

# 4. So why was our earlier factor 1.2?

Because the factor depends on **how much of each group you keep**.

The general factor is:

$$\frac{1-a}{b}$$

where:

- $a$ = fraction of **large-gradient samples retained**
- $1-a$ = fraction of small-gradient samples originally present
- $b$ = fraction of small-gradient samples retained

For our earlier example:

$$a=0.2$$

$$b=0.5$$

Therefore:

$$\frac{1-0.2}{0.5}=\frac{0.8}{0.5}=1.6$$

So if you keep 20% of large-gradient samples and 50% of small-gradient samples, each selected small-gradient sample gets a weight of **1.6**.

The previous example where the factor was $1.2$ corresponded to different sampling proportions.

---

## 5. The important idea

Think about it this way:

```text
Original:
80 small samples
        ↓
Keep only 40
        ↓
Each selected sample must represent
more than one original sample
        ↓
Give each selected sample extra weight
```

So GOSS is essentially saying:

> **"I'm throwing away many small-gradient samples, so I'll increase the contribution of the small-gradient samples that I kept to compensate."**

Large-gradient samples **don't need this correction** because GOSS deliberately keeps all of them.

### One subtle but important point

The goal isn't to make the **sample proportions** look like the original again.

After weighting, you still physically have:

```text
20 large
40 small
```

The goal is to make their **contribution to the gradient/Hessian statistics** approximately represent what we would have obtained from all 100 samples.

That's the reason for the factor.

## LightGBM — EFB (Exclusive Feature Bundling)

### 1. Simple explanation

**EFB = Exclusive Feature Bundling.**

Its purpose is to reduce the **number of features** that LightGBM has to process.

The key observation is:

> Some features are **mutually exclusive** — they are rarely non-zero at the same time.

Such features can be **bundled together into a single feature** without losing much information.

This is especially useful for **high-dimensional sparse datasets**.

---

### 2. What does "exclusive" mean?

Suppose we have three features:

| Row | Feature A | Feature B | Feature C |
|---|---:|---:|---:|
| 1 | 1 | 0 | 0 |
| 2 | 0 | 1 | 0 |
| 3 | 0 | 0 | 1 |
| 4 | 2 | 0 | 0 |
| 5 | 0 | 3 | 0 |

Notice:

```text
Feature A is non-zero → B and C are zero
Feature B is non-zero → A and C are zero
Feature C is non-zero → A and B are zero
```

They are **exclusive**.

Instead of processing:

```text
A
B
C
```

separately, LightGBM can combine them into one **bundled feature**.

---

## 3. Why is this useful?

Suppose your dataset has:

$$10,000$$

features.

But perhaps many of them are sparse:

```text
Feature 1 → mostly 0
Feature 2 → mostly 0
Feature 3 → mostly 0
...
```

If many features are mutually exclusive, processing all 10,000 separately is expensive.

EFB tries to transform:

```text
10,000 original features
          ↓
find mutually exclusive features
          ↓
bundle them
          ↓
2,000 effective features
```

The exact reduction depends on the dataset.

So instead of reducing the **number of samples** like GOSS, EFB reduces the **effective number of features**.

---

## 4. How can different features be put into one feature?

Consider:

```text
A    B    C
1    0    0
0    1    0
0    0    1
2    0    0
0    3    0
```

We can assign different ranges to the features.

For example:

```text
A → values 1–2
B → values 4–6
C → values 8–9
```

Then the bundled feature could become:

```text
A    B    C    Bundle
1    0    0      1
0    1    0      5
0    0    1      8
2    0    0      2
0    3    0      6
```

Because only one of A, B, C is non-zero in each row, their information can occupy different ranges inside the same bundled feature.

The exact encoding is handled internally by LightGBM.

---

## 5. Why does exclusivity matter?

This is the important condition.

Suppose:

```text
Row 1:
A = 2
B = 5
```

Now both are non-zero.

If we simply combine them into one feature, it becomes difficult to determine whether a value came from A or B.

So EFB looks for features that are **approximately mutually exclusive**.

It doesn't require perfect exclusivity in every practical situation; LightGBM can tolerate a limited amount of conflict when creating bundles.

---

## 6. How EFB connects to histogram learning

This is where the concepts connect:

### Without EFB

```text
Many features
     ↓
Histogram construction
     ↓
Evaluate splits for each feature
```

### With EFB

```text
Many sparse features
        ↓
Find mutually exclusive features
        ↓
Bundle them
        ↓
Fewer effective features
        ↓
Histogram construction
        ↓
Evaluate splits
```

So EFB makes the **histogram-based split search cheaper** by reducing the number of feature dimensions that need to be considered.

---

## 7. GOSS vs EFB

This distinction is important:

| Technique | Reduces |
|---|---|
| GOSS | Number of samples used for tree construction |
| EFB | Number of effective features |
| Histogram learning | Number of candidate split values |
| Leaf-wise growth | Number/location of tree expansions |

So LightGBM attacks computational cost from multiple directions.

```text
              LightGBM
                 |
       ┌─────────┼─────────┐
       ↓         ↓         ↓
    GOSS       EFB    Histograms
     ↓           ↓        ↓
 fewer rows   fewer      fewer
              features   thresholds
```

### Core idea to remember

> **EFB takes sparse, mutually exclusive features and bundles them together so LightGBM can process fewer effective features during tree construction.**

Your flow is:

1. **Histogram binning happens first.**
   - Each feature's continuous values are converted into bins.

2. **Find mutually exclusive features.**
   - Example: `A`, `B`, `C` rarely/non-overlap in the same row.

3. **Create one bundle for them.**

4. **Give each feature a separate bin range inside the bundle.**

For example:

```text
A → bundle bins 1–3
B → bundle bins 4–7
C → bundle bins 8–10
```

5. Suppose a row has:

```text
A = bin 2
B = 0
C = 0
```

A's bin 2 is already in A's region, so:

```text
Bundle = 2
```

Suppose another row has:

```text
A = 0
B = bin 3
C = 0
```

B needs to occupy B's region `4–7`.

So LightGBM applies B's offset:

```text
B's original bin = 3
B's offset = 3

3 + 3 = 6
```

Therefore:

```text
Bundle = 6
```

When interpreting that bundled value:

```text
6 → falls in B's range (4–7)
       ↓
therefore B is active
       ↓
remove B's offset
       ↓
6 - 3 = 3
       ↓
original B bin = 3
```

So your mental model is essentially:

```text
Histogram binning
       ↓
Find mutually exclusive features
       ↓
Bundle them
       ↓
Assign separate bin regions
       ↓
Add offsets to move each feature's bins
       ↓
ONE bundled feature
       ↓
Range tells which feature
       ↓
Remove offset → recover that feature's bin
```

**That is the mechanism you should remember for EFB.**

Yes, you **could use a single number**, but that single number would need to encode **both pieces of information**:

1. **Which feature is active**
2. **Which bin/value that feature has**

That's why we use a **range of numbers**.

### Suppose we only use one number per feature

Imagine:

```text
A → 1
B → 2
C → 3
```

Now consider:

```text
A = bin 4
```

How do we represent that?

If we store `1`, we know **A is active**, but we have lost the fact that its bin was `4`.

If we store `4`, we know the bin, but we don't know whether:

```text
A = bin 4
B = bin 4
C = bin 4
```

was active.

So one simple number isn't enough.

---

### The range solves both problems

Suppose:

```text
A → bins 1–5
B → bins 6–10
C → bins 11–15
```

Now bundle value `8` tells us:

```text
8
↓
6–10
↓
B is active
```

And because B's region starts at 6:

$$8-6=2$$

So we recover:

```text
Feature = B
Original bin = 2
```

Therefore:

> **The range acts as the feature identifier, while the position within that range represents the original bin.**

That's why we can't simply say **A = 1, B = 2, C = 3** unless we create some additional encoding mechanism to preserve the original bin information. The range + offset is a convenient way to encode **feature identity + bin identity in one number**.

## LightGBM — Categorical Features

LightGBM can handle **categorical features** directly without necessarily converting them into one-hot encoded columns.

Suppose our dataset has:

| Customer | City | Age | Income |
|---|---|---:|---:|
| C1 | Mumbai | 21 | 45000 |
| C2 | Pune | 35 | 70000 |
| C3 | Mumbai | 28 | 52000 |
| C4 | Delhi | 42 | 90000 |
| C5 | Pune | 31 | 65000 |

Here:

- `Age` → numerical feature
- `Income` → numerical feature
- `City` → **categorical feature**

### The problem with treating categories as numbers

We could encode:

```text
Mumbai → 0
Pune   → 1
Delhi  → 2
```

But now a normal numerical tree might consider splits such as:

```text
City < 1.5
```

which gives:

```text
Mumbai → 0
Pune   → 1
Delhi  → 2
```

and therefore:

```text
{Mumbai, Pune} | {Delhi}
```

The numerical values `0,1,2` themselves have **no natural ordering**.

For example, it doesn't mean:

$$\text{Mumbai} < \text{Pune} < \text{Delhi}$$

So ordinary numerical treatment can be inappropriate.

---

## What LightGBM does

LightGBM can treat `City` as a **categorical feature** rather than treating its encoded values as ordinary continuous numbers.

Conceptually, suppose the categories are:

```text
Mumbai
Pune
Delhi
```

LightGBM evaluates how useful different **category groupings** are for reducing the objective/loss.

For example, a possible split could be:

```text
{Mumbai, Pune} | {Delhi}
```

or:

```text
{Mumbai} | {Pune, Delhi}
```

or:

```text
{Mumbai, Delhi} | {Pune}
```

The important point is:

> The tree is looking for a useful **partition of categories**, rather than assuming that the category numbers have numerical meaning.

### Why this is useful

Without native categorical handling, a categorical feature with many categories may require one-hot encoding:

```text
City_Mumbai
City_Pune
City_Delhi
...
```

This can create **many additional features**, especially when a feature has hundreds or thousands of categories.

LightGBM's categorical handling avoids needing to represent every category as a separate one-hot column.

### Important distinction from EFB

EFB was about:

```text
many sparse FEATURES
        ↓
bundle
        ↓
one feature
```

Categorical handling is about:

```text
one FEATURE
        ↓
contains categories
        ↓
find useful category partition
```

So these are solving **different problems**.

Yes. I stopped too early. **The important part of LightGBM categorical handling is how it actually decides the split.**

### 1. Suppose we have this dataset

Predict whether a customer will buy:

| Customer | City | Buy |
|---|---|---:|
| C1 | Mumbai | 1 |
| C2 | Mumbai | 1 |
| C3 | Mumbai | 0 |
| C4 | Pune | 0 |
| C5 | Pune | 0 |
| C6 | Pune | 1 |
| C7 | Delhi | 0 |
| C8 | Delhi | 0 |
| C9 | Delhi | 0 |

`City` is categorical:

```text
Mumbai
Pune
Delhi
```

We **cannot say**:

```text
Mumbai = 0
Pune   = 1
Delhi  = 2
```

and then simply search for numerical thresholds like `City < 1.5`, because those numbers don't have meaningful ordering.

---

## 2. So how does LightGBM find a split?

The key idea is that LightGBM looks at the **relationship between each category and the target/gradient information**.

For classification, imagine it calculates a statistic representing how strongly each category is associated with the target.

For this simplified example, calculate the average target:

| City | Buy values | Average Buy |
|---|---|---:|
| Mumbai | 1, 1, 0 | 0.67 |
| Pune | 0, 0, 1 | 0.33 |
| Delhi | 0, 0, 0 | 0.00 |

So we get:

```text
Delhi   → 0.00
Pune    → 0.33
Mumbai  → 0.67
```

LightGBM can **order the categories according to their relationship with the target**:

```text
Delhi → Pune → Mumbai
```

Now it can consider boundaries:

```text
Delhi | Pune, Mumbai
```

or

```text
Delhi, Pune | Mumbai
```

Instead of considering every arbitrary numerical ordering.

---

## 3. Why is this useful?

Suppose the best split is:

```text
Delhi, Pune | Mumbai
```

Then the tree effectively does:

```text
             City
              |
       ┌──────┴──────┐
       |             |
 Delhi/Pune       Mumbai
```

It has grouped categories based on their predictive behavior.

Notice that **Mumbai being encoded as 0 or 2 doesn't matter**.

The model is interested in:

> "Which categories should go to the left child and which should go to the right child?"

---

## 4. What about a category with many values?

Suppose:

```text
City =
Mumbai
Pune
Delhi
Nashik
Nagpur
Thane
...
```

One-hot encoding would create:

```text
City_Mumbai
City_Pune
City_Delhi
City_Nashik
City_Nagpur
City_Thane
...
```

Potentially hundreds or thousands of columns.

LightGBM can instead keep:

```text
City
```

as **one categorical feature** and search for useful category partitions.

---

## 5. What is actually being ordered?

This is the important technical detail.

LightGBM doesn't simply calculate the raw target average and call that the final algorithm.

For a boosting iteration, it has **gradients** and **Hessians** for the training samples.

For each category, it can aggregate their gradient/Hessian information and use a category statistic to determine an ordering.

Conceptually:

$$\text{Category statistic} \approx \frac{\sum g_i}{\sum h_i}$$

where:

- $g_i$ = gradient for sample $i$
- $h_i$ = Hessian for sample $i$

The categories are then ordered according to this statistic, and LightGBM searches possible boundaries in that ordering.

So the actual process is closer to:

```text
Categorical feature
       ↓
Group samples by category
       ↓
Calculate gradient/Hessian statistics
       ↓
Order categories
       ↓
Try splits between categories
       ↓
Calculate split gain
       ↓
Choose best categorical split
```

### The main thing to remember

**LightGBM does NOT turn:**

```text
Mumbai = 0
Pune = 1
Delhi = 2
```

into an ordinary numerical feature and blindly use `City < 1.5`.

Instead, it uses the **learning information from the categories** to find a useful ordering/partition.

That's the mechanism that makes LightGBM's categorical handling powerful.

Yes — **that is the basic intuition**, with one important correction.

LightGBM tries to determine **which categories behave similarly with respect to the target**, and then puts categories that belong together into the **same child**.

For example:

| City | Target behavior |
|---|---|
| Mumbai | mostly 1 |
| Pune | mostly 0 |
| Delhi | mostly 0 |

It may find that:

```text
Left child:   Pune + Delhi
Right child:  Mumbai
```

because Pune and Delhi have similar target behavior, while Mumbai behaves differently.

So conceptually:

```text
Categorical feature
       ↓
Look at learning/target information for each category
       ↓
Order/group categories by that information
       ↓
Find the best split
       ↓
Categories with similar behavior → same child
```

### But don't think only "which category has more influence"

It's more accurate to think:

> **Which grouping of categories gives the greatest reduction in the objective/loss?**

For example, even if Mumbai has the strongest positive relationship with the target, LightGBM isn't simply saying:

> "Mumbai has highest influence → put it in one child."

It evaluates possible partitions and chooses the one with the **best split gain**.

So:

**Category behavior → possible grouping → split gain → left/right child.**

That's the key mechanism.

Exactly. **Choosing the missing-value direction does not mean the tree stops there.**

Suppose LightGBM creates:

```text
             Age < 30?
            /         \
         Left         Right
          ↑
       Missing
```

If it learns that **missing values should go Left**, then all missing-Age samples are sent to the Left child **for this split**.

Now the tree continues growing:

```text
                    Age < 30?
                   /         \
              Left            Right
          + missing
             |
          next split
             |
        Income < 50K?
          /       \
       Left       Right
```

The **Left child can absolutely be split again**, just like the Right child.

### Important distinction

The missing-value decision answers only:

> **"For this particular split, which branch should a missing value follow?"**

It does **not** answer:

> "Should this sample become a leaf?"

The tree can continue:

$$\text{Root} \rightarrow \text{Split} \rightarrow \text{Split} \rightarrow \text{Split} \rightarrow \text{Leaf}$$

And at every new split, LightGBM can again determine how missing values should be handled.

For example:

```text
             Age < 30?
            /         \
     Left + NaN       Right
        |
    Income < 50K?
      /       \
     ...      ...
```

So yes, **the purpose of the further splits is to progressively improve the partition/purity/objective**, until the stopping conditions are reached (such as leaf constraints, depth limits, minimum data requirements, etc.).

## LightGBM — Hyperparameters

The main LightGBM hyperparameters you should understand are:

### 1. `num_leaves`

Controls the **maximum number of leaves** in a tree.

```text
Smaller num_leaves
→ simpler tree
→ faster
→ less overfitting

Larger num_leaves
→ more complex tree
→ captures more patterns
→ higher overfitting risk
```

This is particularly important in LightGBM because it uses **leaf-wise growth**.

---

### 2. `max_depth`

Limits how deep the tree can become.

```text
max_depth = 3
```

means a path cannot go deeper than 3 levels.

It can be used to control excessive tree complexity.

---

### 3. `learning_rate`

Controls how strongly each new tree contributes to the model.

The update is conceptually:

$$\hat{y}^{(t)}=\hat{y}^{(t-1)}+\eta f_t(X)$$

where $\eta$ is the learning rate.

Small learning rate:

```text
small update
→ usually needs more trees
→ slower training
→ often better generalization
```

Large learning rate:

```text
large update
→ fewer trees needed
→ faster learning
→ greater overfitting risk
```

---

### 4. `n_estimators`

Number of boosting trees.

For example:

```text
n_estimators = 500
```

means the model can build up to 500 trees.

There is an important relationship:

```text
learning_rate ↓
       ↓
usually increase
n_estimators ↑
```

---

### 5. `min_data_in_leaf`

Minimum number of training samples that a leaf must contain.

Larger value:

```text
fewer tiny leaves
→ simpler model
→ less overfitting
```

Smaller value:

```text
allows very small leaves
→ more complex model
→ higher overfitting risk
```

---

### 6. `feature_fraction`

Fraction of features considered when building each tree.

For example:

```text
feature_fraction = 0.8
```

means approximately **80% of features** are used for a tree.

This introduces randomness and can help reduce overfitting.

---

### 7. `bagging_fraction`

Controls the fraction of training samples used for a boosting iteration when bagging is enabled.

For example:

```text
bagging_fraction = 0.8
```

means approximately 80% of the training data is sampled.

---

### 8. `lambda_l1` and `lambda_l2`

Regularization parameters.

- `lambda_l1` → L1 regularization
- `lambda_l2` → L2 regularization

Increasing regularization generally discourages overly complex models.

---

### 9. `min_gain_to_split`

Minimum gain required for LightGBM to make a split.

If a candidate split gives insufficient improvement:

```text
gain < min_gain_to_split
```

the split is rejected.

So it prevents the tree from making weak/unnecessary splits.

---

### Most important ones to remember

For understanding LightGBM, focus especially on:

| Hyperparameter | Controls |
|---|---|
| `num_leaves` | Tree complexity |
| `max_depth` | Maximum depth |
| `learning_rate` | Contribution of each tree |
| `n_estimators` | Number of trees |
| `min_data_in_leaf` | Minimum samples per leaf |
| `feature_fraction` | Feature sampling |
| `bagging_fraction` | Data sampling |
| `lambda_l1/l2` | Regularization |
| `min_gain_to_split` | Minimum split improvement |

**The most important LightGBM-specific one is `num_leaves`**, because LightGBM's leaf-wise growth makes tree complexity strongly dependent on the number of leaves.

Yes, **exactly**.

If:

```text
bagging_fraction = 0.8
```

and you have 10,000 samples:

```text
Tree 1 → random 8,000 samples
Tree 2 → different random 8,000 samples
Tree 3 → different random 8,000 samples
Tree 4 → different random 8,000 samples
```

So it can look like:

```text
        Training data (10,000)
              ↓
      ┌───────┼───────┐
      ↓       ↓       ↓
    Tree 1  Tree 2  Tree 3
     80%     80%     80%
      ↓       ↓       ↓
    subset  subset  subset
     A        B       C
```

**A, B, and C are randomly selected subsets**, so they can overlap.

For example, sample 152 might appear in Tree 1 and Tree 3 but not Tree 2.

And because LightGBM is boosting, each tree is still trained **sequentially to improve the previous model**; the 80% sampling doesn't turn it into Random Forest.

Yes. On paper, a **LightGBM numerical question** is usually simplified. You generally won't be expected to reproduce the full implementation. The examiner will give a small dataset and ask you to show **histogram construction, leaf-wise split selection, or boosting updates**.

Let's do a complete small example.

## Example: LightGBM on paper

Suppose we have one feature $X$ and target $y$:

| Point | $X$ | $y$ |
|---|---:|---:|
| $P_1$ | 10 | 20 |
| $P_2$ | 20 | 30 |
| $P_3$ | 30 | 25 |
| $P_4$ | 40 | 50 |
| $P_5$ | 50 | 45 |
| $P_6$ | 60 | 70 |

Assume the initial prediction is:

$$\hat{y}_i=40$$

and we are using squared-error loss:

$$l_i=\frac{1}{2}(y_i-\hat{y}_i)^2$$

For squared error:

$$g_i=\hat{y}_i-y_i$$

and

$$h_i=1$$

### Step 1 — Calculate gradients

| Point | $y$ | $\hat y$ | $g=\hat y-y$ | $h$ |
|---|---:|---:|---:|---:|
| P1 | 20 | 40 | 20 | 1 |
| P2 | 30 | 40 | 10 | 1 |
| P3 | 25 | 40 | 15 | 1 |
| P4 | 50 | 40 | -10 | 1 |
| P5 | 45 | 40 | -5 | 1 |
| P6 | 70 | 40 | -30 | 1 |

Notice:

- Positive gradient → model is **overpredicting**
- Negative gradient → model is **underpredicting**

---

# Step 2 — Histogram

Suppose LightGBM creates 3 bins:

| Bin | $X$ values | Points |
|---|---|---|
| B1 | 10–20 | P1, P2 |
| B2 | 30–40 | P3, P4 |
| B3 | 50–60 | P5, P6 |

Now LightGBM doesn't need to examine every individual value. It aggregates gradient and Hessian information inside each bin.

### B1

$$G_1=20+10=30$$

$$H_1=1+1=2$$

### B2

$$G_2=15-10=5$$

$$H_2=1+1=2$$

### B3

$$G_3=-5-30=-35$$

$$H_3=1+1=2$$

So the histogram becomes:

| Bin | $G$ | $H$ |
|---|---:|---:|
| B1 | 30 | 2 |
| B2 | 5 | 2 |
| B3 | -35 | 2 |

This is the key idea of **Histogram-Based Learning**.

---

# Step 3 — Evaluate a split

Suppose we consider:

```text
B1 | B2 + B3
```

Left child:

$$G_L=30$$

$$H_L=2$$

Right child:

$$G_R=5+(-35)=-30$$

$$H_R=2+2=4$$

Parent:

$$G=30+5-35=0$$

$$H=6$$

For simplified XGBoost/LightGBM-style split evaluation, ignoring regularization:

$$\text{Gain}=\frac{1}{2}\left(\frac{G_L^2}{H_L}+\frac{G_R^2}{H_R}-\frac{G^2}{H}\right)$$

Substitute:

$$\text{Gain}=\frac{1}{2}\left(\frac{30^2}{2}+\frac{(-30)^2}{4}-\frac{0^2}{6}\right)$$

$$=\frac{1}{2}(450+225)$$

$$=337.5$$

So this is a strong split.

---

# Step 4 — Another possible split

Try:

```text
B1 + B2 | B3
```

Left:

$$G_L=30+5=35$$

$$H_L=4$$

Right:

$$G_R=-35$$

$$H_R=2$$

Gain:

$$\text{Gain}=\frac{1}{2}\left(\frac{35^2}{4}+\frac{(-35)^2}{2}\right)$$

$$=\frac{1}{2}(306.25+612.5)$$

$$=459.375$$

This split is **better**.

Therefore LightGBM chooses:

```text
          Root
            |
      B1+B2 | B3
       /          \
    Left          Right
```

---

# Step 5 — Why is this LightGBM specifically?

Now comes the important part.

LightGBM uses **leaf-wise growth**.

Suppose after this split we have:

```text
             Root
               |
          B1+B2 | B3
          /          \
       Leaf A       Leaf B
```

LightGBM doesn't necessarily split both leaves equally level-by-level.

It calculates the **best available split for each leaf**.

Suppose:

```text
Leaf A → best possible gain = 80
Leaf B → best possible gain = 20
```

LightGBM chooses:

```text
Leaf A
```

because:

$$80>20$$

So:

```text
             Root
               |
          B1+B2 | B3
          /          \
      Leaf A        Leaf B
        |
      split
```

This is **leaf-wise growth**.

It keeps splitting the leaf that gives the greatest improvement.

---

# Step 6 — Calculate the leaf value

For a leaf, the simplified optimal weight is:

$$w^*=-\frac{G}{H+\lambda}$$

If we use $\lambda=0$:

For the left child:

$$G_L=35,\quad H_L=4$$

so:

$$w_L=-\frac{35}{4}=-8.75$$

For the right child:

$$G_R=-35,\quad H_R=2$$

so:

$$w_R=-\frac{-35}{2}=17.5$$

These are **corrections** to the current prediction.

Current prediction:

$$\hat y=40$$

Therefore:

Left:

$$40-8.75=31.25$$

Right:

$$40+17.5=57.5$$

So after this tree:

```text
B1+B2 → 31.25
B3    → 57.5
```

---

# What you would actually write in an exam

For a small LightGBM numerical problem, follow this order:

### 1. Create bins

```text
Feature values → histogram bins
```

### 2. Calculate $g_i$ and $h_i$

For squared error:

$$g_i=\hat y_i-y_i$$

$$h_i=1$$

### 3. Aggregate each bin

$$G_{\text{bin}}=\sum g_i$$

$$H_{\text{bin}}=\sum h_i$$

### 4. Try candidate splits

For each split calculate:

$$\text{Gain}=\frac{1}{2}\left(\frac{G_L^2}{H_L+\lambda}+\frac{G_R^2}{H_R+\lambda}-\frac{G^2}{H+\lambda}\right)$$

### 5. Select the split with maximum gain.

### 6. For LightGBM, grow **leaf-wise**

Choose the leaf having the **largest available split gain**.

### 7. Calculate leaf output

$$w^*=-\frac{G}{H+\lambda}$$

### 8. Update predictions

$$\hat y_{\text{new}}=\hat y_{\text{old}}+\eta w$$

So the overall exam flow is:

**Dataset → bins → gradients/Hessians → histogram → candidate split gains → best split → leaf-wise growth → leaf weights → prediction update.**

This is the most useful way to solve a **LightGBM numerical question by hand**.